In [11]:
import pandas as pd
import numpy as np
from pandas.api.types import CategoricalDtype
from collections import defaultdict

In [12]:
def preprocess(df):
    # 1. '회' 제거 및 정수형 변환
    count_cols2 = [
        '자발한도감액횟수_R12M',
        '한도증액횟수_R12M',
        '한도심사요청건수'
    ]

    for col in count_cols2:
        if col in df.columns:
            df[col] = (
                df[col].astype(str)
                      .str.replace('회', '', regex=False)
            )
            df[col] = pd.to_numeric(df[col], errors='coerce')  # NaN 그대로 유지

    # 2. 범주형 라벨 인코딩
    le_cols = ['카드론동의여부', 'RV전환가능여부']
    for col in le_cols:
        if col in df.columns:
            codes, _ = pd.factorize(df[col], sort=True)
            df[col] = codes

    return df

df1 = preprocess(pd.read_parquet('train/2.신용정보/201807_train_신용정보.parquet'))
df2 = preprocess(pd.read_parquet('train/2.신용정보/201808_train_신용정보.parquet'))
df3 = preprocess(pd.read_parquet('train/2.신용정보/201809_train_신용정보.parquet'))
df4 = preprocess(pd.read_parquet('train/2.신용정보/201810_train_신용정보.parquet'))
df5 = preprocess(pd.read_parquet('train/2.신용정보/201811_train_신용정보.parquet'))
df6 = preprocess(pd.read_parquet('train/2.신용정보/201812_train_신용정보.parquet'))

In [13]:
dfs = [df.drop(columns=['기준년월'], errors='ignore') for df in [df1, df2, df3, df4, df5, df6]]

def merge_two_avg(df_left, df_right):
    merge_keys = ['ID']
    if 'Segment' in df_left.columns and 'Segment' in df_right.columns:
        merge_keys.append('Segment')

    merged = pd.merge(df_left, df_right, on=merge_keys, how='outer', suffixes=('_left', '_right'))
    result = merged[merge_keys].copy()
    
    # 평균 계산
    for col in set(df_left.columns).union(df_right.columns):
        if col in merge_keys:
            continue
        col_left = f"{col}_left" if f"{col}_left" in merged.columns else None
        col_right = f"{col}_right" if f"{col}_right" in merged.columns else None
        
        cols_to_avg = [c for c in [col_left, col_right] if c is not None]
        result[col] = merged[cols_to_avg].mean(axis=1, skipna=True)
    
    return result

from functools import reduce
merged_df = reduce(merge_two_avg, dfs)
merged_df

,ID,한도심사요청후경과월,카드론동의여부,일시상환론한도금액,CA이자율_할인전,강제한도감액금액_R12M,일시불ONLY전환가능여부,CA한도금액,한도심사거절후경과월,RV전환가능여부,...,월상환론상향가능한도금액,한도증액후경과월,RV최소결제비율,한도증액횟수_R12M,카드이용한도금액_B2M,RV신청일자,CL이자율_할인전,한도심사요청건수,특별한도보유여부_R3M,상향가능한도금액
0,TRAIN_000000,3.0,1.0,0.00000,22.996165,0.00000,0.0,5927.15625,3.0,0.0,...,0.0,12.00000,19.99996,0.0,20685.75000,NaN,18.523273,0.0,0.0,0.00000
1,TRAIN_000001,3.0,1.0,42337.09375,14.810177,0.00000,1.0,4767.43750,3.0,1.0,...,0.0,12.00000,9.99998,0.0,10000.25000,2.017911e+07,15.440785,0.0,0.0,3.03125
2,TRAIN_000002,3.0,1.0,0.00000,22.099407,1.09375,0.0,28294.59375,3.0,0.0,...,0.0,12.00000,19.99996,0.0,77708.71875,2.012041e+07,18.176727,0.0,0.0,0.00000
3,TRAIN_000003,3.0,1.0,0.00000,22.999296,0.00000,0.0,8946.78125,3.0,0.0,...,0.0,12.00000,19.99996,0.0,20942.78125,NaN,22.999938,0.0,0.0,0.00000
4,TRAIN_000004,3.0,1.0,48001.84375,14.741779,0.00000,1.0,52357.75000,3.0,1.0,...,0.0,12.00000,9.99998,0.0,172672.12500,NaN,11.024648,0.0,0.0,0.00000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
399995,TRAIN_399995,3.0,1.0,0.00000,15.159056,0.00000,1.0,9110.34375,3.0,1.0,...,0.0,12.00000,9.99998,0.0,21011.81250,NaN,11.899554,0.0,0.0,3.00000
399996,TRAIN_399996,3.0,1.0,0.00000,14.838240,0.00000,1.0,31092.53125,3.0,1.0,...,0.0,7.03125,9.99998,NaN,79201.18750,NaN,15.580400,0.0,0.0,9.68750
399997,TRAIN_399997,3.0,1.0,0.00000,16.993116,0.00000,0.0,19210.53125,3.0,0.0,...,0.0,12.00000,19.99996,0.0,63230.43750,NaN,17.046863,0.0,0.0,0.00000
399998,TRAIN_399998,3.0,1.0,90002.50000,15.115889,0.25000,1.0,3998.15625,3.0,1.0,...,0.0,12.00000,9.99998,0.0,10001.53125,NaN,11.900438,0.0,0.0,3.50000


In [14]:
df_segment = pd.read_parquet('train/1.회원정보/201807_train_회원정보.parquet')[['ID', 'Segment']]

# 2. 중복 제거 (ID별 Segment가 유일하다는 전제)
df_segment = df_segment.drop_duplicates(subset='ID')

# 3. merged_df에 Segment 열 붙이기 (ID 기준)
merged_df = pd.merge(merged_df, df_segment, on='ID', how='left')
merged_df

,ID,한도심사요청후경과월,카드론동의여부,일시상환론한도금액,CA이자율_할인전,강제한도감액금액_R12M,일시불ONLY전환가능여부,CA한도금액,한도심사거절후경과월,RV전환가능여부,...,한도증액후경과월,RV최소결제비율,한도증액횟수_R12M,카드이용한도금액_B2M,RV신청일자,CL이자율_할인전,한도심사요청건수,특별한도보유여부_R3M,상향가능한도금액,Segment
0,TRAIN_000000,3.0,1.0,0.00000,22.996165,0.00000,0.0,5927.15625,3.0,0.0,...,12.00000,19.99996,0.0,20685.75000,NaN,18.523273,0.0,0.0,0.00000,D
1,TRAIN_000001,3.0,1.0,42337.09375,14.810177,0.00000,1.0,4767.43750,3.0,1.0,...,12.00000,9.99998,0.0,10000.25000,2.017911e+07,15.440785,0.0,0.0,3.03125,E
2,TRAIN_000002,3.0,1.0,0.00000,22.099407,1.09375,0.0,28294.59375,3.0,0.0,...,12.00000,19.99996,0.0,77708.71875,2.012041e+07,18.176727,0.0,0.0,0.00000,C
3,TRAIN_000003,3.0,1.0,0.00000,22.999296,0.00000,0.0,8946.78125,3.0,0.0,...,12.00000,19.99996,0.0,20942.78125,NaN,22.999938,0.0,0.0,0.00000,D
4,TRAIN_000004,3.0,1.0,48001.84375,14.741779,0.00000,1.0,52357.75000,3.0,1.0,...,12.00000,9.99998,0.0,172672.12500,NaN,11.024648,0.0,0.0,0.00000,E
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
399995,TRAIN_399995,3.0,1.0,0.00000,15.159056,0.00000,1.0,9110.34375,3.0,1.0,...,12.00000,9.99998,0.0,21011.81250,NaN,11.899554,0.0,0.0,3.00000,E
399996,TRAIN_399996,3.0,1.0,0.00000,14.838240,0.00000,1.0,31092.53125,3.0,1.0,...,7.03125,9.99998,NaN,79201.18750,NaN,15.580400,0.0,0.0,9.68750,D
399997,TRAIN_399997,3.0,1.0,0.00000,16.993116,0.00000,0.0,19210.53125,3.0,0.0,...,12.00000,19.99996,0.0,63230.43750,NaN,17.046863,0.0,0.0,0.00000,C
399998,TRAIN_399998,3.0,1.0,90002.50000,15.115889,0.25000,1.0,3998.15625,3.0,1.0,...,12.00000,9.99998,0.0,10001.53125,NaN,11.900438,0.0,0.0,3.50000,E


In [15]:
nan_columns = merged_df.columns[merged_df.isnull().any()].tolist()

print("NaN이 포함된 열 목록:")
print(nan_columns)

NaN이 포함된 열 목록:
['한도증액횟수_R12M', 'RV신청일자']


In [16]:
ex1 = merged_df

In [17]:
cols_to_drop = ['한도증액횟수_R12M', 'RV신청일자']
ex1.drop(columns=cols_to_drop, inplace=True)

In [18]:
missing_mask = ex1.isna() | (ex1 == -1)
missing_ratio = missing_mask.mean()

high_na = missing_ratio[missing_ratio > 0.2].index.tolist()

high_const_cols = []
threshold_const = 0.8

for col in ex1.columns:
    top_ratio = ex1[col].value_counts(normalize=True, dropna=False).values[0]
    if top_ratio > threshold_const:
        high_const_cols.append(col)

to_drop = list(set(high_na + high_const_cols))

if 'Segment' in to_drop:
    to_drop.remove('Segment')

print("삭제 대상 컬럼 (결측>20% 또는 동일값>80%):", to_drop)

ex1.drop(columns=to_drop, inplace=True)

삭제 대상 컬럼 (결측>20% 또는 동일값>80%): ['한도심사요청후경과월', '강제한도감액금액_R12M', '한도심사거절후경과월', 'RV전환가능여부', '자발한도감액후경과월', '시장단기연체여부_R3M', '한도요청거절건수', '자발한도감액금액_R12M', '최초한도금액', '시장연체상환여부_R6M', '자발한도감액횟수_R12M', 'rv최초시작후경과일', '시장단기연체여부_R6M', '강제한도감액횟수_R12M', '한도증액금액_R12M', '강제한도감액후경과월', 'RV약정청구율', '시장연체상환여부_R3M', '연체감액여부_R3M', '월상환론상향가능한도금액', '한도증액후경과월', '한도심사요청건수', '특별한도보유여부_R3M']


In [19]:
num_df = ex1.select_dtypes(include=[np.number]).dropna()

corr = num_df.corr().abs()

high_corr_pairs = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        .stack()
        .reset_index()
)
high_corr_pairs.columns = ['Feature_1', 'Feature_2', 'Correlation']
high_corr_pairs = high_corr_pairs[high_corr_pairs['Correlation'] > 0.8]

if not np.issubdtype(ex1['Segment'].dtype, np.number):
    segment_map = {label: idx for idx, label in enumerate(sorted(ex1['Segment'].unique()))}
    ex1['Segment_encoded'] = ex1['Segment'].map(segment_map)
else:
    ex1['Segment_encoded'] = ex1['Segment']

segment_corr = ex1[num_df.columns].corrwith(ex1['Segment_encoded']).abs()

high_corr_pairs['Corr_with_Segment_1'] = high_corr_pairs['Feature_1'].map(segment_corr)
high_corr_pairs['Corr_with_Segment_2'] = high_corr_pairs['Feature_2'].map(segment_corr)

high_corr_pairs = high_corr_pairs.sort_values(by='Correlation', ascending=False).reset_index(drop=True)

print(f"▶ 상관계수 0.7 초과 변수쌍 수: {len(high_corr_pairs)}")
display(high_corr_pairs)


▶ 상관계수 0.7 초과 변수쌍 수: 10


,Feature_1,Feature_2,Correlation,Corr_with_Segment_1,Corr_with_Segment_2
0,카드이용한도금액_B1M,카드이용한도금액_B2M,0.997302,0.306635,0.306025
1,카드이용한도금액_B1M,카드이용한도금액,0.988577,0.306635,0.307779
2,카드이용한도금액,카드이용한도금액_B2M,0.986040,0.307779,0.306025
3,상향가능CA한도금액,상향가능한도금액,0.977603,0.075321,0.060322
4,CA이자율_할인전,RV현금서비스이자율_할인전,0.939574,0.125456,0.094865
5,CA한도금액,카드이용한도금액,0.935391,0.295352,0.307779
6,CA한도금액,카드이용한도금액_B1M,0.932996,0.295352,0.306635
7,CA한도금액,카드이용한도금액_B2M,0.930300,0.295352,0.306025
8,RV현금서비스이자율_할인전,RV일시불이자율_할인전,0.927508,0.094865,0.075702
9,CA이자율_할인전,RV일시불이자율_할인전,0.920497,0.125456,0.075702


In [20]:
to_drop = []

for _, row in high_corr_pairs.iterrows():
    f1, f2 = row['Feature_1'], row['Feature_2']
    c1, c2 = row['Corr_with_Segment_1'], row['Corr_with_Segment_2']
    
    if pd.isna(c1) or pd.isna(c2):
        continue
    
    if c1 < c2:
        to_drop.append(f1)
    else:
        to_drop.append(f2)

to_drop = list(set(to_drop))

# 결과 출력
print(f"▶ 제거 대상 피처 수: {len(to_drop)}")
print("제거할 피처 목록:")
print(to_drop)

▶ 제거 대상 피처 수: 6
제거할 피처 목록:
['카드이용한도금액_B2M', '카드이용한도금액_B1M', 'RV현금서비스이자율_할인전', 'RV일시불이자율_할인전', 'CA한도금액', '상향가능한도금액']


In [21]:
cols_to_drop = ['카드이용한도금액_B2M', '카드이용한도금액_B1M', 'RV현금서비스이자율_할인전', 'RV일시불이자율_할인전', 'CA한도금액', '상향가능한도금액']
ex1.drop(columns=cols_to_drop, inplace=True)

In [22]:
cols_to_drop = ['Segment_encoded']
ex1.drop(columns=cols_to_drop, inplace=True)
cols = ex1.columns.tolist()
cols

['ID',
 '카드론동의여부',
 '일시상환론한도금액',
 'CA이자율_할인전',
 '일시불ONLY전환가능여부',
 '월상환론한도금액',
 '카드이용한도금액',
 '상향가능CA한도금액',
 'RV최소결제비율',
 'CL이자율_할인전',
 'Segment']

In [23]:
ex1.to_parquet('신용_전처리_Segment.parquet', index=False)

In [24]:
ddf1 = preprocess(pd.read_parquet('train/2.신용정보/201807_train_신용정보.parquet'))
ddf2 = preprocess(pd.read_parquet('train/2.신용정보/201808_train_신용정보.parquet'))
ddf3 = preprocess(pd.read_parquet('train/2.신용정보/201809_train_신용정보.parquet'))
ddf4 = preprocess(pd.read_parquet('train/2.신용정보/201810_train_신용정보.parquet'))
ddf5 = preprocess(pd.read_parquet('train/2.신용정보/201811_train_신용정보.parquet'))
ddf6 = preprocess(pd.read_parquet('train/2.신용정보/201812_train_신용정보.parquet'))

In [25]:

dfs = [ddf1, ddf2, ddf3, ddf4, ddf5, ddf6]
for i in range(len(dfs)):
    if '기준년월' in dfs[i].columns:
        dfs[i] = dfs[i].drop(columns=['기준년월'])

# ID 기준으로 병합 후 평균
from functools import reduce

merged_df = reduce(
    lambda left, right: pd.merge(left, right, on='ID', how='outer', suffixes=('', '_dup')),
    dfs
)

# 같은 이름의 열 평균 구하기
from collections import defaultdict
import pandas as pd

result = pd.DataFrame()
result['ID'] = merged_df['ID']

# 열 이름 모아 평균 구하기
col_dict = defaultdict(list)
for col in merged_df.columns:
    if col != 'ID':
        base_col = col.split('_dup')[0]
        col_dict[base_col].append(col)

for base_col, cols in col_dict.items():
    result[base_col] = merged_df[cols].mean(axis=1, skipna=True)

# 결과 확인
print(result.head())

             ID  최초한도금액       카드이용한도금액        CA한도금액     일시상환론한도금액  \
0  TRAIN_000000     0.0   19654.961538   6114.807692      0.000000   
1  TRAIN_000001     0.0    9996.384615   5093.769231  42187.346154   
2  TRAIN_000002     0.0   87369.538462  29288.153846      0.000000   
3  TRAIN_000003     0.0   18955.269231   8394.461538      0.000000   
4  TRAIN_000004     0.0  176097.000000  51494.769231  48002.115385   

        월상환론한도금액  CA이자율_할인전  CL이자율_할인전  RV일시불이자율_할인전  RV현금서비스이자율_할인전  ...  \
0       0.000000  22.995685  18.455541     18.033014       21.216460  ...   
1   91925.615385  14.804079  15.238810     10.818499       13.646867  ...   
2       0.000000  21.961916  18.237277     17.350416       21.232852  ...   
3       0.000000  22.998153  22.999935     19.388842       22.998433  ...   
4  155153.076923  14.717841  11.091033     10.439197       10.538200  ...   

   연체감액여부_R3M  한도심사요청건수  한도요청거절건수  한도심사요청후경과월  한도심사거절후경과월  시장단기연체여부_R6M  \
0         0.0       0.0       0.0        

In [27]:
cols = ['ID',
 '카드론동의여부',
 '일시상환론한도금액',
 'CA이자율_할인전',
 '일시불ONLY전환가능여부',
 '월상환론한도금액',
 '카드이용한도금액',
 '상향가능CA한도금액',
 'RV최소결제비율',
 'CL이자율_할인전',]

result = result[cols]
result

,ID,카드론동의여부,일시상환론한도금액,CA이자율_할인전,일시불ONLY전환가능여부,월상환론한도금액,카드이용한도금액,상향가능CA한도금액,RV최소결제비율,CL이자율_할인전
0,TRAIN_000000,1.0,0.000000,22.995685,0.0,0.000000,19654.961538,0.000000,19.99996,18.455541
1,TRAIN_000001,1.0,42187.346154,14.804079,1.0,91925.615385,9996.384615,1.000000,9.99998,15.238810
2,TRAIN_000002,1.0,0.000000,21.961916,0.0,0.000000,87369.538462,0.000000,19.99996,18.237277
3,TRAIN_000003,1.0,0.000000,22.998153,0.0,0.000000,18955.269231,0.000000,19.99996,22.999935
4,TRAIN_000004,1.0,48002.115385,14.717841,1.0,155153.076923,176097.000000,0.000000,9.99998,11.091033
...,...,...,...,...,...,...,...,...,...,...
399995,TRAIN_399995,1.0,0.000000,14.990697,1.0,54223.692308,20088.500000,1.000000,9.99998,11.898756
399996,TRAIN_399996,1.0,0.000000,14.844236,1.0,156357.538462,86333.846154,3.000000,9.99998,15.427747
399997,TRAIN_399997,1.0,0.000000,16.967790,0.0,0.000000,52560.230769,0.000000,19.99996,16.340850
399998,TRAIN_399998,1.0,90001.692308,14.957319,1.0,180858.115385,10002.346154,1.000000,9.99998,11.899652


In [28]:
result.to_parquet('신용_전처리_test.parquet', index=False)

In [29]:
nan_columns = result.columns[result.isnull().any()].tolist()

print("NaN이 포함된 열 목록:")
print(nan_columns)

NaN이 포함된 열 목록:
[]
